In [10]:
!pip install -q -U \
    transformers==4.46.3 \
    peft==0.13.2 \
    trl==0.11.4 \
    accelerate \
    datasets \
    bitsandbytes


In [11]:
import os
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


CUDA available: True
GPU: Tesla T4
GPU memory (GB): 15.64


In [12]:
raw_dataset = load_dataset("tatsu-lab/alpaca", split="train")
print(raw_dataset)
raw_dataset[0]


Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 52002
})


{'instruction': 'Give three tips for staying healthy.',
 'input': '',
 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.',
 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}

In [13]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def to_chat_text(example):
    user_content = example["instruction"]
    if example["input"].strip():
        user_content += "\n\n" + example["input"]

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}


# Use a manageable subset so a full run fits inside a free-tier GPU session's time limit;
# increase `NUM_EXAMPLES` if you have more session time available.
NUM_EXAMPLES = 3000
dataset = raw_dataset.shuffle(seed=SEED).select(range(NUM_EXAMPLES))
dataset = dataset.map(to_chat_text, remove_columns=dataset.column_names)

split = dataset.train_test_split(test_size=0.05, seed=SEED)
train_dataset, eval_dataset = split["train"], split["test"]

print(train_dataset)
print(eval_dataset)
print(train_dataset[0]["text"])


Dataset({
    features: ['text'],
    num_rows: 2850
})
Dataset({
    features: ['text'],
    num_rows: 150
})
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Update the number in the cell.

The cell in the spreadsheet contains the number "3".<|im_end|>
<|im_start|>assistant
The cell has been updated to contain the number "4".<|im_end|>



In [14]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # NormalFloat4 — matched to ~normal weight distributions
    bnb_4bit_use_double_quant=True,       # quantize the quantization constants too
    bnb_4bit_compute_dtype=torch.float16 # dtype used for the on-the-fly dequantized matmul
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False  # incompatible with gradient checkpointing during training

print(model)


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)


In [15]:
# Report the actual GPU memory footprint of the 4-bit base model
if torch.cuda.is_available():
    print("GPU memory allocated for 4-bit base model (GB):",
          round(torch.cuda.memory_allocated() / 1e9, 3))


GPU memory allocated for 4-bit base model (GB): 1.982


In [16]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [17]:
training_args = SFTConfig(
    output_dir="./qlora_qwen_checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch size = 4 * 4 = 16
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    bf16=False,
    fp16=True,
    optim="paged_adamw_8bit",   # paged optimizer keeps optimizer states memory-friendly on free-tier GPUs
    report_to="none",
    max_seq_length=512,
    dataset_text_field="text",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)


/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


In [18]:
train_result = trainer.train()
print(train_result)


Step,Training Loss,Validation Loss
100,1.229100,1.248498
200,1.223500,1.238408
300,1.211000,1.234087


TrainOutput(global_step=356, training_loss=1.2582875570554413, metrics={'train_runtime': 850.7307, 'train_samples_per_second': 6.7, 'train_steps_per_second': 0.418, 'total_flos': 6738015161825280.0, 'train_loss': 1.2582875570554413, 'epoch': 1.997194950911641})


In [19]:
trainer.state.log_history

[{'loss': 1.7956,
  'grad_norm': 0.38471853733062744,
  'learning_rate': 0.0001991884797578617,
  'epoch': 0.1402524544179523,
  'step': 25},
 {'loss': 1.2974,
  'grad_norm': 0.40890008211135864,
  'learning_rate': 0.00019375990128440204,
  'epoch': 0.2805049088359046,
  'step': 50},
 {'loss': 1.2409,
  'grad_norm': 0.3627493977546692,
  'learning_rate': 0.00018349313988777914,
  'epoch': 0.42075736325385693,
  'step': 75},
 {'loss': 1.2291,
  'grad_norm': 0.3967190086841583,
  'learning_rate': 0.00016891797929200375,
  'epoch': 0.5610098176718092,
  'step': 100},
 {'eval_loss': 1.2484983205795288,
  'eval_runtime': 7.0298,
  'eval_samples_per_second': 21.338,
  'eval_steps_per_second': 5.406,
  'epoch': 0.5610098176718092,
  'step': 100},
 {'loss': 1.2419,
  'grad_norm': 0.4796972870826721,
  'learning_rate': 0.00015078652452418063,
  'epoch': 0.7012622720897616,
  'step': 125},
 {'loss': 1.2357,
  'grad_norm': 0.5683653354644775,
  'learning_rate': 0.00013003439191525807,
  'epoch': 

In [20]:
ADAPTER_DIR = "./qwen_qlora_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(os.listdir(ADAPTER_DIR))


['tokenizer_config.json', 'adapter_model.safetensors', 'README.md', 'special_tokens_map.json', 'merges.txt', 'tokenizer.json', 'added_tokens.json', 'vocab.json', 'adapter_config.json']


In [21]:
# Free GPU memory from the 4-bit training run
del trainer, model
torch.cuda.empty_cache() if torch.cuda.is_available() else None


In [22]:
base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto" if torch.cuda.is_available() else None,
)

merged_model = PeftModel.from_pretrained(base_model_fp16, ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()
merged_model = merged_model.to(torch.float16)

print(type(merged_model))


<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>


In [23]:
MERGED_DIR = "./qwen_merged_fp16"
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(os.listdir(MERGED_DIR))


['tokenizer_config.json', 'generation_config.json', 'special_tokens_map.json', 'merges.txt', 'config.json', 'tokenizer.json', 'added_tokens.json', 'vocab.json', 'model.safetensors']


In [24]:
!zip -r qwen_merged_fp16.zip qwen_merged_fp16 > /dev/null
print("Zipped merged model for download.")


Zipped merged model for download.


In [25]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MERGED_DIR = "./qwen_merged_fp16"

device = torch.device("cpu")

# On CPU, float32 is generally faster/more numerically robust than float16
# (most CPU kernels don't have optimized fp16 matmul paths), so we upcast for inference.
cpu_model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.float32)
cpu_model.to(device)
cpu_model.eval()

cpu_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)

def generate(prompt, max_new_tokens=200):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]
    text = cpu_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = cpu_tokenizer(text, return_tensors="pt").to(device)

    start = time.time()
    with torch.no_grad():
        output_ids = cpu_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=cpu_tokenizer.eos_token_id,
        )
    elapsed = time.time() - start

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    response = cpu_tokenizer.decode(generated, skip_special_tokens=True)
    print(f"[{elapsed:.1f}s, {len(generated)} tokens, "
          f"{len(generated)/elapsed:.2f} tok/s]\n{response}\n")
    return response

_ = generate("Give three tips for staying healthy.")
_ = generate("Explain what a LoRA adapter is in one paragraph.")
_ = generate("Write a short poem about the ocean.")


[16.4s, 53 tokens, 3.23 tok/s]
1. Eat a balanced diet with plenty of fruits and vegetables.
2. Exercise regularly, aiming to get at least 30 minutes of moderate exercise per day.
3. Get enough sleep each night in order to feel refreshed and energized the next day.

[33.1s, 108 tokens, 3.27 tok/s]
A LoRA (Low-Rank Adaptation) adapter is a type of neural network architecture that allows for faster and more efficient training of large models with limited computational resources. It works by reducing the size of the model's weights, allowing it to be trained on less data and using fewer compute cycles. The resulting model has lower precision than larger models but still performs well on tasks such as image recognition and language translation. LoRA adapters can also be used to reduce the complexity of pre-trained models so they can be deployed quickly and efficiently.

[37.4s, 124 tokens, 3.32 tok/s]
The ocean is vast and deep, 
A place of mystery and power.
It washes away all pain,
And br